In [1]:
import sqlite3
import pandas as pd
import numpy as np

In [12]:


conn = sqlite3.connect("yelp.db")

query = """
WITH restaurant_base AS (
    SELECT
        business_id,
        name,
        city,
        state,
        stars AS avg_rating,
        review_count
    FROM business
    WHERE categories LIKE '%Restaurant%'
),

review_summary AS (
    SELECT
        business_id,
        COUNT(*) AS total_reviews
    FROM review
    GROUP BY business_id
),

tip_summary AS (
    SELECT
        business_id,
        COUNT(*) AS total_tips
    FROM tip
    GROUP BY business_id
),

checkin_summary AS (
    SELECT
        business_id,
        SUM(
            LENGTH(date) - LENGTH(REPLACE(date, ',', '')) + 1
        ) AS total_checkins
    FROM checkin
    GROUP BY business_id
),

restaurant_metrics AS (
    SELECT
        r.business_id,
        r.name,
        r.city,
        r.state,
        r.avg_rating,
        r.review_count,
        COALESCE(rv.total_reviews,0) AS total_reviews,
        COALESCE(t.total_tips,0) AS total_tips,
        COALESCE(c.total_checkins,0) AS total_checkins
    FROM restaurant_base r
    LEFT JOIN review_summary rv
        ON r.business_id = rv.business_id
    LEFT JOIN tip_summary t
        ON r.business_id = t.business_id
    LEFT JOIN checkin_summary c
        ON r.business_id = c.business_id
)

SELECT
    business_id,
    name,
    city,
    state,
    avg_rating,
    review_count,
    total_reviews,
    total_tips,
    total_checkins,

    CASE
        WHEN avg_rating >= 4 THEN 'High Rated'
        WHEN avg_rating >= 3 THEN 'Medium Rated'
        ELSE 'Low Rated'
    END AS rating_segment

FROM restaurant_metrics
"""

df = pd.read_sql_query(query, conn)

conn.close()



In [14]:
# get summary table for visulization
df.to_csv("yelp_summary_dataset.csv", index=False)


In [5]:
# city matrix 
city_metrics = df.groupby(["city","state"]).agg(
    restaurant_count=("business_id","count"),
    avg_city_rating=("avg_rating","mean"),
    total_city_reviews=("review_count","sum")
).reset_index()

city_metrics.to_csv("city_metrics.csv", index=False)